# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Build a chat model client for any OpenAI-compatible endpoint.
2. Define a **tool** — a plain Python function — with the `@tool` decorator.
3. Create an agent with `create_agent` and run it.
4. Stream the agent's response token-by-token.

## Setup

Prerequisites: run `pip install -r requirements.txt` in the repository root, copy `.env.example` to `.env`, fill in `LLM_BASE_URL`, `LLM_API_KEY`, `LLM_MODEL`, and run `python scripts/check_endpoint.py`.

The cell below loads those variables from `.env` and builds the chat model client. `ChatOpenAI` speaks the OpenAI Chat Completions protocol, so the same code works with DeepSeek, OpenAI, a local Ollama server, or any other compatible endpoint — only the three environment variables change. `LLM_EXTRA_BODY` carries optional provider-specific request options (for DeepSeek it turns thinking mode off).

In [1]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

missing = [name for name in ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL") if not os.environ.get(name)]
if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Copy .env.example to .env in the repository root and fill them in."
    )

llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ["LLM_API_KEY"],
    extra_body=json.loads(os.environ.get("LLM_EXTRA_BODY") or "null"),
)
print(f"Model client ready: {os.environ['LLM_MODEL']} @ {os.environ['LLM_BASE_URL']}")

Model client ready: claude-3-5-sonnet-20241022 @ http://127.0.0.1:8045/v1


## Creating Your First Agent

An agent needs two things:

- **Instructions** that tell it *who it is* and *how to behave*. In LangChain these become the agent's **system prompt**.
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions. The function's **docstring becomes the tool description** the model reads when deciding whether to call it, and its type hints define the arguments.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.

In [2]:
from langchain.tools import tool


@tool
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona", "Paris", "Berlin", "Tokyo", "Sydney",
        "New York City", "Cairo", "Cape Town", "Rio de Janeiro", "Bali",
    ]

Now we wire the model, the tool and the instructions together with `create_agent`. The agent runs a **tool-calling loop**: it sends the conversation to the model, executes any tool the model asks for, feeds the result back, and repeats until the model answers in plain text.

`agent.invoke` returns the full message history — user message, the model's tool call, the tool result, and the final reply. `reply_text` pulls the text out of the last message (some providers return content as a list of blocks rather than a single string).

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    llm,
    tools=[get_destinations],
    system_prompt=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)


def reply_text(result) -> str:
    """Return the text of the last message; content can be a string or a list of blocks."""
    content = result["messages"][-1].content
    if isinstance(content, list):
        return "".join(block.get("text", "") for block in content if isinstance(block, dict))
    return content


result = agent.invoke(
    {"messages": [{"role": "user", "content": "I'm looking for a warm beach destination. What do you recommend?"}]}
)
print(reply_text(result))

Based on our available destinations, here are my top recommendations for a **warm beach vacation**:

🌴 **Bali, Indonesia**
Bali is a paradise for beach lovers! It offers stunning tropical beaches, crystal-clear waters, lush scenery, and a vibrant culture. It's perfect if you're looking for a mix of relaxation, adventure, and unique cultural experiences — all in a warm, tropical climate.

🌊 **Rio de Janeiro, Brazil**
Rio is famous for its iconic beaches like **Copacabana** and **Ipanema**. It boasts warm weather, a lively atmosphere, breathtaking scenery (think Sugarloaf Mountain!), and a rich, vibrant culture. It's a fantastic choice if you love a buzzing beach city vibe.

🌞 **Barcelona, Spain**
If you prefer a European flavor, Barcelona has beautiful Mediterranean beaches with warm summers. Plus, you get the bonus of world-class architecture, amazing food, and a lively nightlife scene.

🦁 **Cape Town, South Africa**
Cape Town offers gorgeous beaches along with dramatic mountain landsc

## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

`agent.astream(..., stream_mode="messages")` yields `(token, metadata)` pairs for every message chunk the graph produces. We print only the chunks that come from the model node (tool calls and tool results also flow through the stream). Jupyter supports top-level `await`, so the `async for` below runs as-is.

In [4]:
async for token, metadata in agent.astream(
    {"messages": [{"role": "user", "content": "Tell me about Tokyo as a travel destination"}]},
    stream_mode="messages",
):
    if metadata.get("langgraph_node") == "model" and getattr(token, "content", None):
        print(token.content, end="", flush=True)
print()

Let

 me look

 up our

 available destinations to get

 you information

...

 help

 planning a

 trip there

?

😊

## Summary

In this lesson you learned how to:

- **Create a model client** with `ChatOpenAI`, which talks to any OpenAI-compatible endpoint — the provider is just configuration.
- **Define a tool** with the `@tool` decorator, which turns a plain Python function (and its docstring) into something the model can call.
- **Create an agent** with `create_agent`, which wires the model, tools and instructions into a tool-calling loop.
- **Stream responses** with `astream` to print tokens as they arrive.

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.